# Build the background-flow cache

Run this notebook once before `background_relative_beta_effect.ipynb`. It uses the current Jupyter kernel, avoiding differences between the notebook environment and the shell Python.

The builder is restartable: completed monthly-file partitions are skipped if the notebook is interrupted and rerun.

In [1]:
# Main user settings
N_WORKERS = 4       # Start with 4; more workers may become I/O-limited
ANNULUS_INNER_RC = 1.5
ANNULUS_OUTER_RC = 3.0

In [2]:
from pathlib import Path
import sys

HERE = Path.cwd()
if not (HERE / 'background_flow_tools.py').exists():
    HERE = Path('MRes/seacofs_eddy_tilt_analysis/beta_effect_background_flow').resolve()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import seacofs_tilt_tools as tilt
from background_flow_tools import BackgroundConfig, build_background_cache

print('Python:', sys.version.split()[0])
print('Workers:', N_WORKERS)
if sys.version_info < (3, 9):
    raise RuntimeError('Select a Jupyter kernel running Python 3.9 or newer.')

Python: 3.10.8
Workers: 4


## Load and select eddies

The cache is built for all eddy observations. The topographic PV-gradient calculation retains `(w + f)`, but planetary/topographic dominance is deliberately filtered only in the analysis notebook. This makes the expensive cache reusable for alternative thresholds and subsets.

In [3]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
eddies = tilt.add_pv_gradient_terms(eddies, grid)
selected = eddies.copy()
print(f"Selected {len(selected):,} observations from {selected['Eddy'].nunique():,} eddies")

Selected 127,426 observations from 2,982 eddies


## Build or resume the cache

This is the long-running cell on the first run. It reads each archive file once, accumulates the monthly and full-archive climatologies, and extracts cropped annulus backgrounds for matching eddy days. On rerun, completed file partitions are skipped and their saved sums/counts are re-reduced without reopening the NetCDF archive.

In [4]:
config = BackgroundConfig(
    annulus_inner_rc=ANNULUS_INNER_RC,
    annulus_outer_rc=ANNULUS_OUTER_RC,
)
background = build_background_cache(
    selected, grid, config=config, workers=N_WORKERS
)
print('Cache complete:', config.background_table_path)
print(f"Rows: {len(background):,}; eddies: {background['Eddy'].nunique():,}")

Required levels by depth: {200: 30, 500: 30}; reading 30/30 surface sigma levels.
All sigma levels are required because sampled shallow columns lie entirely above the depth limit.


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   19.0s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   29.8s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   48.7s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:  1.1min
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:  1.5min
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  1.8min
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:  2.4min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  2.8min
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:  3.4min
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:  4.0min
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:  4.6min
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:  5.4min
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:  6.2min
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  7.0min
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:  8.0min
[Parallel(

Cache complete: /srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/background_flow_cache_all_eddies_v3/eddy_day_background.parquet
Rows: 127,426; eddies: 2,982


When the final cell finishes, open and run `background_relative_beta_effect.ipynb`. If this notebook is interrupted, rerun it with the same settings; completed monthly files will be reused.